# 4 Nearest Neighbors
### Team Members: June, Anh, Tara, Jackson

# Data Cleaning

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("train.csv", low_memory=False)
test_df = pd.read_csv("test.csv", low_memory=False)

missing_placeholders = ["UNKNOWN", "ZZZZ"]

train_df = train_df.replace(missing_placeholders, np.nan)
test_df = test_df.replace(missing_placeholders, np.nan)

print(train_df.shape)
train_df.head()


(307178, 55)


,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,...,NUM_SEEN,NUM_STRUCK,SIZE,ENROUTE_STATE,COMMENTS,SOURCE,PERSON,LUPDATE,TRANSFER,INDICATED_DAMAGE
0,1410120,12/13/93,12,1993,NaN,Day,TJSJ,LUIS MUNOZ MARIN INTL,18.43942,-66.00183,...,10-Feb,10-Feb,Small,NaN,NaN,FAA Form 5200-7,Pilot,4/3/23,0,0
1,709688,2/1/10,2,2010,5:00,Night,WMKK,KUALA LUMPUR INTL,2.745578,101.709917,...,NaN,1,Medium,NaN,2010-5-18-53374 /Legacy Record 300758/,FAA Form 5200-7-E,Air Transport Operations,6/9/10,0,0
2,730841,5/9/12,5,2012,2:00,Night,KSDF,MUHAMMAD ALI INTERNATIONAL,38.17439,-85.736,...,NaN,1,Large,NaN,UPS EVENT REPT 36216 (4/22/13 UPDATED COST) /L...,Air Transport Report,Air Transport Operations,4/22/13,0,1
3,654676,10/8/02,10,2002,NaN,NaN,KLAX,LOS ANGELES INTL,33.94254,-118.40807,...,NaN,10-Feb,Medium,NaN,2002-10-8-111929 /Legacy Record 216397/,FAA Form 5200-7-E,Carcass Found,1/9/03,0,0
4,629708,2/3/97,2,1997,NaN,Dawn,PHLI,LIHUE ARPT,21.97598,-159.33896,...,1,1,Medium,NaN,SOURCE 5200-7 & PACIR /Legacy Record 121531/,Multiple,NaN,3/1/07,0,0


Drop columns that are not useful or have a high missing value fraction.

In [2]:
missing_summary = (
    train_df.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_fraction")
)

missing_summary.head(20)

high_missing_cols = missing_summary[missing_summary["missing_fraction"] > 0.95].index.tolist()
high_missing_cols


manual_drop_cols = [
    "BIRD_BAND_NUMBER",   # almost entirely missing and not useful for prediction
    "REMARKS",            # free-text notes, difficult to use in simple models
    "COMMENTS",           # free-text notes, difficult to use in simple models
    "LUPDATE",            # administrative update date, not strike condition
    "TRANSFER",           # administrative field
    "REG",
    "LOCATION"            # Dropped b/c free text, not standardized
]

target_col = "INDICATED_DAMAGE"

cols_to_drop = sorted(set(high_missing_cols + manual_drop_cols))

X = train_df.drop(columns=[target_col] + cols_to_drop, errors="ignore")
y = train_df[target_col]

X_test = test_df.drop(columns=cols_to_drop, errors="ignore")

print("Dropped columns:")
print(cols_to_drop)

print("Training features shape:", X.shape)
print("Test features shape:", X_test.shape)




Dropped columns:
['BIRD_BAND_NUMBER', 'COMMENTS', 'ENG_3_POS', 'ENG_4_POS', 'ENROUTE_STATE', 'LOCATION', 'LUPDATE', 'PRECIPITATION', 'REG', 'REMARKS', 'TRANSFER']
Training features shape: (307178, 43)
Test features shape: (34131, 43)


Drop training rows where the majority of the remaining feature values are missing. We use a 50% threshold because this directly matches our definition of "majority missing information" and removes only the most incomplete records. We do not drop rows from `test_df` because Kaggle requires a prediction for every test row.

In [3]:
row_missing_fraction = X.isna().mean(axis=1)

row_missing_summary = row_missing_fraction.describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
print(row_missing_summary)

missing_row_threshold = 0.50
rows_to_keep = row_missing_fraction <= missing_row_threshold

rows_removed = (~rows_to_keep).sum()
percent_removed = rows_removed / len(X) * 100

X = X.loc[rows_to_keep].copy()
y = y.loc[rows_to_keep].copy()

print(f"Rows removed: {rows_removed} ({percent_removed:.2f}% of training data)")
print("Cleaned training features shape:", X.shape)
print("Cleaned target shape:", y.shape)


count    307178.000000
mean          0.217663
std           0.178880
min           0.000000
50%           0.139535
75%           0.441860
90%           0.465116
95%           0.465116
99%           0.488372
max           0.674419
dtype: float64
Rows removed: 332 (0.11% of training data)
Cleaned training features shape: (306846, 43)
Cleaned target shape: (306846,)


Impute the remaining missing values. Numerical features are filled with the training mean, and categorical features are filled with the training mode. The same values learned from `X` are applied to `X_test`.

In [4]:
X_test = X_test.reindex(columns=X.columns)

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.columns.difference(numeric_cols).tolist()

numeric_fill_values = X[numeric_cols].mean().fillna(0)
categorical_fill_values = X[categorical_cols].agg(
    lambda col: col.mode(dropna=True).iloc[0]
    if not col.mode(dropna=True).empty
    else "Missing"
)

missing_before = pd.DataFrame(
    {
        "train_missing": X.isna().sum(),
        "test_missing": X_test.isna().sum(),
    }
)

X[numeric_cols] = X[numeric_cols].fillna(numeric_fill_values)
X_test[numeric_cols] = X_test[numeric_cols].fillna(numeric_fill_values)

X[categorical_cols] = X[categorical_cols].fillna(categorical_fill_values)
X_test[categorical_cols] = X_test[categorical_cols].fillna(categorical_fill_values)

print(f"Numeric columns imputed with mean: {len(numeric_cols)}")
print(f"Categorical columns imputed with mode: {len(categorical_cols)}")
print("Missing values before imputation:")
print(missing_before[missing_before.sum(axis=1) > 0].head(20))
print("Missing values after imputation:")
print({"train_missing": int(X.isna().sum().sum()), "test_missing": int(X_test.isna().sum().sum())})


Numeric columns imputed with mean: 15
Categorical columns imputed with mode: 28
Missing values before imputation:
             train_missing  test_missing
TIME                109103         12219
TIME_OF_DAY         133680         14958
AIRPORT_ID           40376          4493
AIRPORT              40376          4493
LATITUDE             40436          4510
LONGITUDE            40439          4510
RUNWAY               75254          8315
STATE                40436          4510
FAAREGION            40436          4510
OPERATOR             86613          9751
FLT                 161157         18010
AIRCRAFT             86664          9798
AMA                  88598         10002
AMO                 116500         13151
EMA                 102094         11510
EMO                 112848         12667
AC_CLASS             87022          9855
AC_MASS              87122          9849
TYPE_ENG             87455          9897
NUM_ENGS             87366          9887
Missing values after impu

{'train_missing': 0, 'test_missing': 0}


Make Sure there are no duplicates

In [5]:
duplicate_count = train_df.duplicated().sum()
print("Duplicate Count: ", duplicate_count)

Duplicate Count:  0
